# 🏦 Bank Customer Churn Analysis — Python EDA

**Project:** Bank Customer Churn Analysis  
**Purpose:** Python EDA that reproduces the business logic from `Bank_Customer_Churn_Business_Insights.sql`.

### SQL → Python workflow
- Load the cleaned Excel dataset
- Validate data quality
- Calculate the same KPIs as the SQL file
- Analyze geography, demographics, products, engagement and financial behavior
- Build high-risk customer segments and risk scores
- Generate charts for Power BI/reporting
- Finish with actionable business recommendations

**Target:** `Exited` — 1 = Churned, 0 = Retained.


In [ ]:
# 1. Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("Libraries loaded successfully.")


## 2. Load the cleaned dataset

The notebook first looks for `Bank_Customer_Churn_Cleaning_Dataset.xlsx` in the working directory and then `/mnt/data/`.

For analysis, use the **Clean_Reference** sheet created with the project dataset. If it is not available, the notebook falls back to `Raw_Data` so the workflow can still be inspected.


In [ ]:
# Locate the Excel dataset
possible_files = [
    Path("Bank_Customer_Churn_Cleaning_Dataset.xlsx"),
    Path("/mnt/data/Bank_Customer_Churn_Cleaning_Dataset.xlsx")
]

excel_path = next((p for p in possible_files if p.exists()), None)

if excel_path is None:
    raise FileNotFoundError(
        "Place Bank_Customer_Churn_Cleaning_Dataset.xlsx in the notebook folder "
        "or /mnt/data before running this cell."
    )

xls = pd.ExcelFile(excel_path)
print("File:", excel_path)
print("Sheets:", xls.sheet_names)

sheet = "Clean_Reference" if "Clean_Reference" in xls.sheet_names else "Raw_Data"
df = pd.read_excel(excel_path, sheet_name=sheet)

print(f"Loaded sheet: {sheet}")
print("Shape:", df.shape)
display(df.head())


## 3. Data quality checks — SQL Q01–Q03 equivalent

In [ ]:
# Q01 — Total rows and unique customers
print("Total rows:", len(df))
print("Unique customers:", df["CustomerID"].nunique())

# Q02 — Duplicate CustomerIDs
duplicates = df[df.duplicated("CustomerID", keep=False)].sort_values("CustomerID")
print("\nDuplicate CustomerIDs:", duplicates["CustomerID"].nunique())
display(duplicates.head(20))

# Q03 — Missing values
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)

quality = pd.DataFrame({
    "missing_count": missing,
    "missing_pct": missing_pct.round(2)
})
display(quality)


## 4. Data types and descriptive statistics

In [ ]:
# Standardize numeric columns for analysis
numeric_cols = [
    "CustomerID", "CreditScore", "Age", "Tenure", "Balance",
    "NumOfProducts", "HasCrCard", "IsActiveMember",
    "EstimatedSalary", "Exited"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print(df.dtypes)
display(df.describe(include="all").T)


## 5. Executive KPIs — SQL Q04–Q06

In [ ]:
total_customers = len(df)
churned_customers = int((df["Exited"] == 1).sum())
retained_customers = int((df["Exited"] == 0).sum())

churn_rate = churned_customers / total_customers * 100
retention_rate = retained_customers / total_customers * 100

total_balance = df["Balance"].sum()
churned_balance = df.loc[df["Exited"] == 1, "Balance"].sum()

kpis = pd.Series({
    "Total Customers": total_customers,
    "Churned Customers": churned_customers,
    "Retained Customers": retained_customers,
    "Churn Rate %": churn_rate,
    "Retention Rate %": retention_rate,
    "Average Credit Score": df["CreditScore"].mean(),
    "Average Age": df["Age"].mean(),
    "Average Balance": df["Balance"].mean(),
    "Average Salary": df["EstimatedSalary"].mean(),
    "Total Balance": total_balance,
    "Churned Balance": churned_balance
})

display(kpis.to_frame("Value"))


In [ ]:
# Churned vs retained — SQL Q05
status_summary = (
    df.assign(Customer_Status=np.where(df["Exited"].eq(1), "Churned", "Retained"))
      .groupby("Customer_Status")
      .size()
      .reset_index(name="Customers")
)
status_summary["Pct_of_Customers"] = (
    status_summary["Customers"] / total_customers * 100
).round(2)

display(status_summary)


## 6. Churn distribution

In [ ]:
plt.figure(figsize=(7, 5))
counts = df["Exited"].map({0: "Retained", 1: "Churned"}).value_counts()
counts.plot(kind="bar")
plt.title("Customer Churn Distribution")
plt.xlabel("Customer Status")
plt.ylabel("Customers")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 7. Geography analysis — SQL Q07–Q08

In [ ]:
geo_analysis = (
    df.groupby("Geography", dropna=False)
      .agg(
          Customers=("CustomerID", "count"),
          Churned=("Exited", "sum")
      )
      .reset_index()
)
geo_analysis["Churn_Rate_%"] = (
    geo_analysis["Churned"] / geo_analysis["Customers"] * 100
).round(2)
geo_analysis["Share_of_Total_Churn_%"] = (
    geo_analysis["Churned"] / churned_customers * 100
).round(2)

display(geo_analysis.sort_values("Churn_Rate_%", ascending=False))

plt.figure(figsize=(8, 5))
plt.bar(geo_analysis["Geography"].astype(str), geo_analysis["Churn_Rate_%"])
plt.title("Churn Rate by Geography")
plt.xlabel("Geography")
plt.ylabel("Churn Rate (%)")
plt.tight_layout()
plt.show()


## 8. Gender and Geography × Gender — SQL Q09–Q10

In [ ]:
def churn_summary(data, group_cols):
    out = (
        data.groupby(group_cols, dropna=False)
            .agg(Customers=("CustomerID", "count"), Churned=("Exited", "sum"))
            .reset_index()
    )
    out["Churn_Rate_%"] = (out["Churned"] / out["Customers"] * 100).round(2)
    return out.sort_values("Churn_Rate_%", ascending=False)

gender_analysis = churn_summary(df, ["Gender"])
geo_gender = churn_summary(df, ["Geography", "Gender"])

display(gender_analysis)
display(geo_gender)


In [ ]:
plt.figure(figsize=(7, 5))
plt.bar(gender_analysis["Gender"].astype(str), gender_analysis["Churn_Rate_%"])
plt.title("Churn Rate by Gender")
plt.xlabel("Gender")
plt.ylabel("Churn Rate (%)")
plt.tight_layout()
plt.show()


## 9. Age segmentation — SQL Q11

In [ ]:
df["Age_Group"] = pd.cut(
    df["Age"],
    bins=[17, 24, 34, 44, 54, np.inf],
    labels=["18-24", "25-34", "35-44", "45-54", "55+"]
)

age_analysis = churn_summary(df, ["Age_Group"])
display(age_analysis)

plt.figure(figsize=(8, 5))
plt.bar(age_analysis["Age_Group"].astype(str), age_analysis["Churn_Rate_%"])
plt.title("Churn Rate by Age Group")
plt.xlabel("Age Group")
plt.ylabel("Churn Rate (%)")
plt.tight_layout()
plt.show()


## 10. Credit score segmentation — SQL Q12

In [ ]:
df["Credit_Band"] = pd.cut(
    df["CreditScore"],
    bins=[-np.inf, 579, 669, 739, 799, np.inf],
    labels=["Poor", "Fair", "Good", "Very Good", "Excellent"]
)

credit_analysis = churn_summary(df, ["Credit_Band"])
display(credit_analysis)

plt.figure(figsize=(9, 5))
plt.bar(credit_analysis["Credit_Band"].astype(str), credit_analysis["Churn_Rate_%"])
plt.title("Churn Rate by Credit Score Band")
plt.xlabel("Credit Score Band")
plt.ylabel("Churn Rate (%)")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## 11. Tenure analysis — SQL Q13–Q14

In [ ]:
tenure_analysis = churn_summary(df, ["Tenure"])
display(tenure_analysis.sort_values("Tenure"))

df["Tenure_Group"] = pd.cut(
    df["Tenure"],
    bins=[-np.inf, 2, 5, np.inf],
    labels=["0-2 Years", "3-5 Years", "6-10 Years"]
)

tenure_group_analysis = churn_summary(df, ["Tenure_Group"])
display(tenure_group_analysis)

plt.figure(figsize=(9, 5))
plt.plot(
    tenure_analysis["Tenure"],
    tenure_analysis["Churn_Rate_%"],
    marker="o"
)
plt.title("Churn Rate by Tenure")
plt.xlabel("Tenure (Years)")
plt.ylabel("Churn Rate (%)")
plt.xticks(sorted(tenure_analysis["Tenure"].dropna().unique()))
plt.tight_layout()
plt.show()


## 12. Product, credit-card and activity analysis — SQL Q15–Q18

In [ ]:
product_analysis = churn_summary(df, ["NumOfProducts"])
card_analysis = churn_summary(df, ["HasCrCard"])
activity_analysis = churn_summary(df, ["IsActiveMember"])
product_activity = churn_summary(df, ["NumOfProducts", "IsActiveMember"])

display(product_analysis)
display(card_analysis)
display(activity_analysis)
display(product_activity)


In [ ]:
fig_data = product_analysis.sort_values("NumOfProducts")

plt.figure(figsize=(8, 5))
plt.bar(fig_data["NumOfProducts"].astype(str), fig_data["Churn_Rate_%"])
plt.title("Churn Rate by Number of Products")
plt.xlabel("Number of Products")
plt.ylabel("Churn Rate (%)")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 5))
labels = activity_analysis["IsActiveMember"].map({0: "Inactive", 1: "Active"}).astype(str)
plt.bar(labels, activity_analysis["Churn_Rate_%"])
plt.title("Churn Rate: Active vs Inactive Members")
plt.xlabel("Membership Status")
plt.ylabel("Churn Rate (%)")
plt.tight_layout()
plt.show()


## 13. Financial analysis — SQL Q19–Q21

In [ ]:
financial_profile = (
    df.assign(Status=np.where(df["Exited"].eq(1), "Churned", "Retained"))
      .groupby("Status")
      .agg(
          Customers=("CustomerID", "count"),
          Avg_Balance=("Balance", "mean"),
          Avg_Salary=("EstimatedSalary", "mean"),
          Avg_Credit_Score=("CreditScore", "mean"),
          Avg_Age=("Age", "mean")
      )
      .reset_index()
)

display(financial_profile)

df["Balance_Band"] = pd.cut(
    df["Balance"],
    bins=[-0.01, 0, 50000, 100000, 150000, np.inf],
    labels=["Zero Balance", "< 50K", "50K-100K", "100K-150K", "150K+"]
)

df["Salary_Band"] = pd.cut(
    df["EstimatedSalary"],
    bins=[-np.inf, 50000, 100000, 150000, np.inf],
    labels=["< 50K", "50K-100K", "100K-150K", "150K+"]
)

balance_analysis = churn_summary(df, ["Balance_Band"])
salary_analysis = churn_summary(df, ["Salary_Band"])

display(balance_analysis)
display(salary_analysis)


In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(balance_analysis["Balance_Band"].astype(str), balance_analysis["Churn_Rate_%"])
plt.title("Churn Rate by Balance Band")
plt.xlabel("Balance Band")
plt.ylabel("Churn Rate (%)")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 5))
plt.bar(salary_analysis["Salary_Band"].astype(str), salary_analysis["Churn_Rate_%"])
plt.title("Churn Rate by Salary Band")
plt.xlabel("Salary Band")
plt.ylabel("Churn Rate (%)")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## 14. Correlation analysis

In [ ]:
corr_cols = [
    "CreditScore", "Age", "Tenure", "Balance",
    "NumOfProducts", "HasCrCard", "IsActiveMember",
    "EstimatedSalary", "Exited"
]
corr = df[corr_cols].corr()

print("Correlation with Exited:")
display(corr["Exited"].sort_values(ascending=False).to_frame("Correlation"))

plt.figure(figsize=(9, 7))
plt.imshow(corr, aspect="auto")
plt.colorbar(label="Correlation")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
plt.yticks(range(len(corr.index)), corr.index)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()


## 15. High-risk customer segments — SQL Q22–Q26

In [ ]:
# Q22 — Older + inactive
older_inactive = df[(df["Age"] >= 45) & (df["IsActiveMember"] == 0)]
display(pd.DataFrame({
    "Customers": [len(older_inactive)],
    "Churned": [older_inactive["Exited"].sum()],
    "Churn_Rate_%": [older_inactive["Exited"].mean() * 100]
}).round(2))

# Q23 — Inactive + high balance
inactive_high_balance = df[
    (df["IsActiveMember"] == 0) & (df["Balance"] >= 100000)
]
display(pd.DataFrame({
    "Customers": [len(inactive_high_balance)],
    "Churned": [inactive_high_balance["Exited"].sum()],
    "Churn_Rate_%": [inactive_high_balance["Exited"].mean() * 100],
    "Avg_Balance": [inactive_high_balance["Balance"].mean()]
}).round(2))

# Q24 — Inactive + multiple products
inactive_multi = df[
    (df["IsActiveMember"] == 0) & (df["NumOfProducts"] >= 2)
]
display(churn_summary(inactive_multi, ["NumOfProducts"]))

# Q25 — High-value churned customers
high_value_churned = (
    df[df["Exited"] == 1]
    [["CustomerID","Surname","Geography","Age","CreditScore","Balance",
      "NumOfProducts","IsActiveMember","EstimatedSalary"]]
    .sort_values("Balance", ascending=False)
    .head(25)
)
display(high_value_churned)


## 16. Risk scoring — SQL Q26, Q32–Q33

In [ ]:
# Build the same simple business risk score used in the SQL file.
df["Risk_Score"] = (
    (df["Age"] >= 45).astype(int)
    + (df["CreditScore"] < 600).astype(int)
    + (df["Balance"] >= 100000).astype(int)
    + (df["NumOfProducts"] >= 2).astype(int)
    + (df["IsActiveMember"] == 0).astype(int)
)

risk_analysis = churn_summary(df, ["Risk_Score"])
display(risk_analysis.sort_values("Risk_Score"))

plt.figure(figsize=(9, 5))
plt.plot(
    risk_analysis["Risk_Score"],
    risk_analysis["Churn_Rate_%"],
    marker="o"
)
plt.title("Churn Rate by Business Risk Score")
plt.xlabel("Risk Score")
plt.ylabel("Churn Rate (%)")
plt.xticks(sorted(risk_analysis["Risk_Score"].dropna().unique()))
plt.tight_layout()
plt.show()

# Retained customers with 3+ risk factors = retention opportunity
retention_targets = df[
    (df["Exited"] == 0) & (df["Risk_Score"] >= 3)
]

print("Potential retention targets:", len(retention_targets))
print("Target customer balance:", round(retention_targets["Balance"].sum(), 2))
print("Average target balance:", round(retention_targets["Balance"].mean(), 2))
display(
    retention_targets[
        ["CustomerID","Geography","Age","CreditScore","Balance",
         "NumOfProducts","IsActiveMember","Risk_Score"]
    ].sort_values(["Risk_Score","Balance"], ascending=[False, False]).head(50)
)


## 17. Combined high-risk segments — SQL Q27–Q30

In [ ]:
geo_activity = churn_summary(df, ["Geography", "IsActiveMember"])
geo_age = churn_summary(df, ["Geography", "Age_Group"])
gender_activity = churn_summary(df, ["Gender", "IsActiveMember"])

print("Geography × Activity")
display(geo_activity[geo_activity["Customers"] >= 10])

print("Geography × Age Group")
display(geo_age[geo_age["Customers"] >= 10])

print("Gender × Activity")
display(gender_activity)

# Management-style segment table
df["Age_Segment"] = pd.cut(
    df["Age"],
    bins=[-np.inf, 34, 49, np.inf],
    labels=["Under 35", "35-49", "50+"]
)

management_segments = churn_summary(
    df,
    ["Geography", "Gender", "Age_Segment", "IsActiveMember"]
)
management_segments = management_segments[
    management_segments["Customers"] >= 15
]

display(management_segments.head(20))


## 18. Balance at risk — SQL Q31

In [ ]:
balance_risk = (
    df.groupby("Geography")
      .agg(
          Total_Balance=("Balance", "sum"),
          Churned_Balance=("Balance", lambda x: x[df.loc[x.index, "Exited"].eq(1)].sum())
      )
      .reset_index()
)
balance_risk["Balance_at_Risk_%"] = (
    balance_risk["Churned_Balance"] / balance_risk["Total_Balance"] * 100
).round(2)

display(balance_risk.sort_values("Balance_at_Risk_%", ascending=False))


## 19. Top churned customers by priority value — SQL Q34

In [ ]:
top_churned = df[df["Exited"] == 1].copy()
top_churned["Priority_Value"] = (
    top_churned["Balance"] + top_churned["EstimatedSalary"] * 0.10
)

top_churned = top_churned.sort_values(
    "Priority_Value", ascending=False
).head(20)

display(
    top_churned[
        ["CustomerID","Geography","Age","CreditScore","Balance",
         "EstimatedSalary","NumOfProducts","IsActiveMember","Priority_Value"]
    ]
)


## 20. Automated business insights

The following section turns the analysis into portfolio-ready statements. Exact values are generated from the dataset when the notebook runs.


In [ ]:
# Generate concise insights from the actual data
highest_geo = geo_analysis.loc[geo_analysis["Churn_Rate_%"].idxmax()]
highest_age = age_analysis.loc[age_analysis["Churn_Rate_%"].idxmax()]
highest_product = product_analysis.loc[product_analysis["Churn_Rate_%"].idxmax()]
highest_risk = risk_analysis.loc[risk_analysis["Churn_Rate_%"].idxmax()]
active_churn = activity_analysis.loc[
    activity_analysis["IsActiveMember"] == 1, "Churn_Rate_%"
].iloc[0]
inactive_churn = activity_analysis.loc[
    activity_analysis["IsActiveMember"] == 0, "Churn_Rate_%"
].iloc[0]

print("BUSINESS INSIGHTS")
print("-" * 60)
print(
    f"1. Overall churn rate is {churn_rate:.2f}% "
    f"({churned_customers:,} of {total_customers:,} customers)."
)
print(
    f"2. {highest_geo['Geography']} has the highest geography-level churn rate "
    f"at {highest_geo['Churn_Rate_%']:.2f}%."
)
print(
    f"3. Age group {highest_age['Age_Group']} has the highest churn rate "
    f"at {highest_age['Churn_Rate_%']:.2f}%."
)
print(
    f"4. Product segment {highest_product['NumOfProducts']} has the highest "
    f"observed churn rate at {highest_product['Churn_Rate_%']:.2f}%."
)
print(
    f"5. Inactive members churn at {inactive_churn:.2f}% versus "
    f"{active_churn:.2f}% for active members."
)
print(
    f"6. Customers with the highest business risk score show a churn rate "
    f"of {highest_risk['Churn_Rate_%']:.2f}%."
)
print(
    f"7. Churned customers hold approximately {churned_balance:,.2f} "
    f"in account balance."
)
print(
    "8. Recommended action: prioritize inactive/high-balance/high-risk "
    "customers for retention campaigns and relationship-manager outreach."
)


## 21. Portfolio conclusion

### Business problem
The bank wants to reduce customer churn and identify segments requiring retention efforts.

### Analysis performed
- Customer and churn KPIs
- Geography and demographic segmentation
- Age and credit-score analysis
- Tenure analysis
- Product and engagement analysis
- Balance and salary segmentation
- High-value churn analysis
- Risk scoring
- Balance-at-risk analysis

### Recommended business actions
1. Prioritize high-risk and inactive customers.
2. Create targeted retention campaigns for high-balance customers.
3. Investigate regions and demographic segments with unusually high churn.
4. Review customers with unusual product combinations.
5. Use the Power BI dashboard to monitor churn KPIs and segment changes.

### Project stack
**Excel → SQL → Python → Power BI → Business Insights → GitHub**
